In [38]:
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import KNNImputer,SimpleImputer,IterativeImputer

In [ ]:
training_faults_diagnostics_df = pd.read_csv(
    filepath_or_buffer='../data/training_faults_diagnostics.csv',
    low_memory=False
)
training_faults_diagnostics_df

In [ ]:
training_faults_diagnostics_df.info()

# Impute missing values


In [21]:
missing = training_faults_diagnostics_df.isnull().mean() * 100
print(missing.sort_values(ascending=False))

ServiceDistance              99.979680
SwitchedBatteryVoltage       89.680541
FuelTemperature              74.205463
ParkingBrake                 67.806164
Throttle                     66.316280
FuelLevel                    58.423789
AcceleratorPedal             55.776230
CruiseControlActive          51.754470
CruiseControlSetSpeed        51.629998
EngineTimeLtd                51.148933
TurboBoostPressure           50.939873
Speed                        50.889025
EngineOilTemperature         50.883165
FuelLtd                      50.801507
FuelRate                     50.779392
EngineLoad                   50.751322
DistanceLtd                  50.714556
BarometricPressure           50.709075
EngineCoolantTemperature     50.703499
EngineOilPressure            50.690078
IntakeManifoldTemperature    50.688944
EngineRpm                    50.646791
IgnStatus                    48.634352
ecuSoftwareVersion           21.414104
eventDescription              5.267520
ecuModel                 

In [33]:
df = training_faults_diagnostics_df.copy()

# Column groups


num_cols = df.select_dtypes(include=["int64", "float64"]).columns

drop_cols = df[num_cols].columns[df[num_cols].isnull().mean() > 0.8]

 # Remove them
df = df.drop(columns=drop_cols)

In [34]:
num_cols = df.select_dtypes(include=["int64", "float64"]).columns
cat_cols = df.select_dtypes(include=["object", "bool"]).columns

# split numeric further 
low_missing = df[num_cols].columns[df[num_cols].isnull().mean() <= 0.4]
mid_missing = df[num_cols].columns[(df[num_cols].isnull().mean() > 0.4) &
                                   (df[num_cols].isnull().mean() <= 0.8)]

#Categorical 

df[cat_cols] = SimpleImputer(strategy="most_frequent").fit_transform(df[cat_cols])


# Low missing numeric 
df[low_missing] = SimpleImputer(strategy="median").fit_transform(df[low_missing])

In [ ]:
# # Mid missing numeric 

# scaler = StandardScaler()
# scaled = scaler.fit_transform(df[mid_missing])

# knn = KNNImputer(n_neighbors=2, weights="distance")
# df[mid_missing] = scaler.inverse_transform(knn.fit_transform(scaled))

# training_faults_diagnostics_df = df

## Imputation on service columns(num columns) if misssing values >0.4 and <0.8 by IterativeImputer

In [40]:
# Mid missing numeric 

#Iterativeimputer uses multiple features to estimate missing values, capturing complex relationships between columns
iter_imputer = IterativeImputer(max_iter=15,random_state=30)

df[mid_missing] = iter_imputer.fit_transform(df[mid_missing])
training_faults_diagnostics_df = df


In [43]:
missing = training_faults_diagnostics_df.isnull().mean() * 100
print(missing.sort_values(ascending=False))

RecordID                     0.0
FuelLtd                      0.0
EngineCoolantTemperature     0.0
EngineLoad                   0.0
EngineOilPressure            0.0
EngineOilTemperature         0.0
EngineRpm                    0.0
EngineTimeLtd                0.0
FuelLevel                    0.0
FuelRate                     0.0
EventTimeStamp               0.0
FuelTemperature              0.0
IgnStatus                    0.0
IntakeManifoldTemperature    0.0
LampStatus                   0.0
ParkingBrake                 0.0
Speed                        0.0
Throttle                     0.0
DistanceLtd                  0.0
CruiseControlSetSpeed        0.0
CruiseControlActive          0.0
BarometricPressure           0.0
eventDescription             0.0
ecuSoftwareVersion           0.0
ecuModel                     0.0
ecuMake                      0.0
ecuSource                    0.0
spn                          0.0
fmi                          0.0
active                       0.0
activeTran

In [44]:
training_faults_diagnostics_df

,RecordID,EventTimeStamp,eventDescription,ecuSoftwareVersion,ecuModel,ecuMake,ecuSource,spn,fmi,active,...,FuelLtd,FuelRate,FuelTemperature,IgnStatus,IntakeManifoldTemperature,LampStatus,ParkingBrake,Speed,Throttle,TurboBoostPressure
0,1.0,2015-02-21 10:47:13,Low (Severity Low) Engine Coolant Level,unknown,unknown,unknown,0.0,111.0,17.0,True,...,12300.907429,0.000000,64.135157,False,78.800000,1023.0,True,0.000000,91.078406,0.000000
1,2.0,2015-02-21 11:34:34,Low (Severity Low) Engine Coolant Level,unknown,unknown,unknown,11.0,629.0,12.0,True,...,50266.879395,4.315023,36.717387,True,106.268387,1279.0,False,24.095211,69.085078,5.827316
2,3.0,2015-02-21 11:35:31,Incorrect Data Steering Wheel Angle,unknown,unknown,unknown,11.0,1807.0,2.0,False,...,50266.879395,4.315023,36.717387,True,106.268387,1279.0,False,24.095211,69.085078,5.827316
3,4.0,2015-02-21 11:35:33,Incorrect Data Steering Wheel Angle,unknown,unknown,unknown,11.0,1807.0,2.0,True,...,50266.879395,4.315023,36.717387,True,106.268387,1279.0,False,24.095211,69.085078,5.827316
4,5.0,2015-02-21 11:39:41,Low (Severity Low) Engine Coolant Level,22281684P01*22357957P01*22362082P01*,0USA13_13_0415_2238A,VOLVO,0.0,4364.0,17.0,False,...,50266.879395,4.315023,36.717387,True,106.268387,16639.0,False,24.095211,69.085078,5.827316
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1058064,1246276.0,2011-01-01 00:05:27,Low Voltage (Particulate Trap Outlet Pressure 1),05317106*05088449*051718172255*09401583*G1*BDR*,6X1u13D1500000000,CMMNS,0.0,3610.0,4.0,False,...,50266.879395,4.315023,36.717387,True,106.268387,18431.0,False,24.095211,69.085078,5.827316
1058065,1246277.0,2011-01-01 00:05:27,Low (Severity Low) Engine Coolant Level,05317106*05088449*051718172255*09401583*G1*BDR*,6X1u13D1500000000,CMMNS,0.0,5742.0,9.0,False,...,50266.879395,4.315023,36.717387,True,106.268387,18431.0,False,24.095211,69.085078,5.827316
1058066,1246278.0,2011-01-01 00:05:27,Low Voltage (Aftertreatment 1 Particulate Trap...,05317106*05088449*051718172255*09401583*G1*BDR*,6X1u13D1500000000,CMMNS,0.0,3251.0,4.0,False,...,50266.879395,4.315023,36.717387,True,106.268387,18431.0,False,24.095211,69.085078,5.827316
1058067,1246279.0,2011-01-01 00:05:27,Incorrect Data Catalyst Dosing Unit,05317106*05088449*051718172255*09401583*G1*BDR*,6X1u13D1500000000,CMMNS,0.0,3361.0,2.0,False,...,50266.879395,4.315023,36.717387,True,106.268387,18431.0,False,24.095211,69.085078,5.827316


In [52]:
training_faults_diagnostics_df[training_faults_diagnostics_df['ecuModel']== 'unknown']

,RecordID,EventTimeStamp,eventDescription,ecuSoftwareVersion,ecuModel,ecuMake,ecuSource,spn,fmi,active,...,FuelLtd,FuelRate,FuelTemperature,IgnStatus,IntakeManifoldTemperature,LampStatus,ParkingBrake,Speed,Throttle,TurboBoostPressure
0,1.0,2015-02-21 10:47:13,Low (Severity Low) Engine Coolant Level,unknown,unknown,unknown,0.0,111.0,17.0,True,...,12300.907429,0.000000,64.135157,False,78.800000,1023.0,True,0.000000,91.078406,0.000000
1,2.0,2015-02-21 11:34:34,Low (Severity Low) Engine Coolant Level,unknown,unknown,unknown,11.0,629.0,12.0,True,...,50266.879395,4.315023,36.717387,True,106.268387,1279.0,False,24.095211,69.085078,5.827316
2,3.0,2015-02-21 11:35:31,Incorrect Data Steering Wheel Angle,unknown,unknown,unknown,11.0,1807.0,2.0,False,...,50266.879395,4.315023,36.717387,True,106.268387,1279.0,False,24.095211,69.085078,5.827316
3,4.0,2015-02-21 11:35:33,Incorrect Data Steering Wheel Angle,unknown,unknown,unknown,11.0,1807.0,2.0,True,...,50266.879395,4.315023,36.717387,True,106.268387,1279.0,False,24.095211,69.085078,5.827316
6,7.0,2015-02-21 11:40:52,Low (Severity Low) Engine Coolant Level,unknown,unknown,unknown,0.0,111.0,17.0,True,...,40961.065437,14.291750,25.568860,True,78.800000,1023.0,False,41.534780,86.097763,20.590000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1058044,1230887.0,2000-05-23 19:17:57,Abnormal Frequency J1939 Network #2,unknown,unknown,unknown,49.0,1231.0,8.0,True,...,50077.560519,4.295157,36.718495,True,108.410824,1279.0,True,22.221012,64.892569,5.752995
1058045,1230888.0,2000-05-23 19:21:20,Abnormal Update Rate Headway Controller Forwar...,unknown,unknown,unknown,49.0,886.0,9.0,False,...,50266.879395,4.315023,36.717387,True,106.268387,65535.0,False,24.095211,69.085078,5.827316
1058046,1230889.0,2000-05-23 19:21:20,Abnormal Frequency J1939 Network #2,unknown,unknown,unknown,49.0,1231.0,8.0,False,...,50266.879395,4.315023,36.717387,True,106.268387,65535.0,False,24.095211,69.085078,5.827316
1058047,1230890.0,2000-05-23 19:21:20,Abnormal Update Rate Source Address of Control...,unknown,unknown,unknown,49.0,1482.0,9.0,False,...,50266.879395,4.315023,36.717387,True,106.268387,65535.0,False,24.095211,69.085078,5.827316
